## Experiment 1: Grayscale Kernel Variants



The objective of this experiment is to compare three different implementations of RGB-to-grayscale conversion:

1. A pure Python loop implementation.
2. A vectorized PyTorch implementation.
3. A custom CUDA kernel.

The experiment is intended to study how the same computation behaves when executed through sequential Python code, optimized tensor operations, and explicit GPU parallelism.

The comparison will focus on:

* Correctness of the generated grayscale output.
* Execution latency.
* Effective memory bandwidth.
* The performance effect of removing Python-level loops.
* How a simple one-thread-per-pixel CUDA mapping performs for a low-arithmetic-intensity, memory-bandwidth-sensitive workload.

The custom CUDA kernel will assign one GPU thread to each output pixel and compute:

$$
Y_i = 0.21R_i + 0.72G_i + 0.07B_i
$$

The experiment should help establish the relationship between parallelism, memory traffic, and performance for a simple CUDA kernel before moving to more complex workloads such as matrix multiplication.


## Implementation

In [1]:
!nvidia-smi

Fri Sep 11 01:21:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ninja

In [3]:
#importing libraries
import torch # standard torch operations
from torch.utils.cpp_extension import load_inline # loading inline C++/CUDA code
import pandas as pd # data manipulation
import time # timing operations
from matplotlib import pyplot as plt # plotting

print("Library Versions:")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"cuDNN: {torch.backends.cudnn.version()}")
print(f"Pandas: {pd.__version__}")

Library Versions:
PyTorch: 2.11.0+cu128
CUDA: True
cuDNN: 91900
Pandas: 2.2.3


In [4]:
# lets generate a random image input
image_size = 1024
# Creating a CHW image over CPU for the python loop implementation and torch vectorized implementation
input_image = torch.randint(0, 255, (3, image_size, image_size), device='cpu')

In [5]:
print(input_image.shape)

torch.Size([3, 1024, 1024])


In [6]:
# python loop implementation of grayscale 
def rgb_to_grayscale_python(input_image):
    # extract dimensions of the input image
    c, h, w = input_image.shape
    # flatten the image to a 2D array
    input_image_flattened = input_image.view(c, h * w)
    # use the for loop to compute the grayscale image
    grayscale_image = torch.zeros(
        (h, w), 
        dtype=input_image.dtype, 
        device='cpu'
    )
    for i in range(h):
        for j in range(w):
            grayscale_image[i, j] = 0.21 * input_image_flattened[0, i * w + j] + 0.72 * input_image_flattened[1, i * w + j] + 0.07 * input_image_flattened[2, i * w + j]
    return grayscale_image

# pytorch loop implementation of grayscale 
def rgb_to_grayscale_pytorch(input_image):
    return 0.21 * input_image[0] + 0.72 * input_image[1] + 0.07 * input_image[2]

In [7]:
help(load_inline)

Help on function load_inline in module torch.utils.cpp_extension:

load_inline(
    name,
    cpp_sources,
    cuda_sources=None,
    sycl_sources=None,
    functions=None,
    extra_cflags=None,
    extra_cuda_cflags=None,
    extra_sycl_cflags=None,
    extra_ldflags=None,
    extra_include_paths=None,
    build_directory=None,
    verbose=False,
    with_cuda=None,
    with_sycl=None,
    is_python_module=True,
    with_pytorch_error_handling=True,
    keep_intermediates=True,
    use_pch=False,
    no_implicit_headers=False
)
    Load a PyTorch C++ extension just-in-time (JIT) from string sources.

    This function behaves exactly like :func:`load`, but takes its sources as
    strings rather than filenames. These strings are stored to files in the
    build directory, after which the behavior of :func:`load_inline` is
    identical to :func:`load`.

    See `the
    tests <https://github.com/pytorch/pytorch/blob/master/test/test_cpp_extensions_jit.py>`_
    for good examples of u

In [8]:
cpp_code = """
#include <torch/extension.h>

torch::Tensor rgb_to_grayscale_cuda(torch::Tensor input_image);

"""

cuda_code = """
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cstdint>

__global__ void rgb_to_grayscale_cuda_kernel(
    const uint8_t* input_image,
    uint8_t* grayscale_image,
    int c,
    int h,
    int w
){
    // Output is a combination of the three channels of the input image
    // Implement Bounds Checking
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < h * w){
        // Compute the grayscale value
        float grayscale_value = 0.21 * input_image[idx] + 0.72 * input_image[idx + h * w] + 0.07 * input_image[idx + 2 * h * w];
        // Store the grayscale value
        grayscale_image[idx] = static_cast<uint8_t>(grayscale_value);
    }
}

torch::Tensor rgb_to_grayscale_cuda(torch::Tensor input_image){

    // Step 1: Implement Basic Checks
    // 1.1 Check if input image is on the GPU
    TORCH_CHECK(input_image.device().is_cuda(), "Input image must be on the GPU");
    // 1.2 Check if input image is contiguous
    TORCH_CHECK(input_image.is_contiguous(), "Input image must be contiguous");
    // 1.3 Check if input image is a uint8 tensor
    TORCH_CHECK(input_image.dtype() == torch::kUInt8, "Input image must be a uint8 tensor");
    // 1.4 Check if input image has 3 channels and three dimensions only (CHW format)
    TORCH_CHECK(input_image.dim() == 3 && input_image.size(0) == 3, "Input image must have 3 channels and three dimensions only (CHW format)");
    
    // Step 2: Extract Dimensions
    // 2.1 Extract dimensions of the input image
    int c = input_image.size(0);
    int h = input_image.size(1);
    int w = input_image.size(2);
    
    // Step 3: Allocate Output Image
    // 3.1 Allocate output image of dimensions (h, w)
    torch::Tensor grayscale_image = torch::zeros({h, w}, input_image.options());
    
    // Step 4: Launch CUDA Kernel with a standard block size of 256
    int block_size = 256;
    int num_blocks = (h * w + block_size - 1) / block_size;

    // 4.1 Launch CUDA kernel with dimensions (h, w)
    rgb_to_grayscale_cuda_kernel<<<num_blocks, block_size>>>(
        input_image.data_ptr<uint8_t>(),
        grayscale_image.data_ptr<uint8_t>(),
        c,
        h,
        w
    );

    C10_CUDA_KERNEL_LAUNCH_CHECK();
    
    return grayscale_image;
}


"""







In [9]:
grayscale_module = load_inline(
    name="grayscale_v3",
    cpp_sources=cpp_code,
    cuda_sources=cuda_code,
    functions=["rgb_to_grayscale_cuda"],
    extra_cflags=["-O3"],
    extra_cuda_cflags=["-O3"],
    verbose=True
)

In [21]:
def benchmarking_run(input_image, cuda_runs=10, cuda_warmup=10):
    # run the benchmark
    start_python_time = time.time()
    rgb_to_grayscale_python(input_image)
    end_python_time = time.time()
    elapsed_avg_python_time = (end_python_time - start_python_time)
    print(f"Average Python Time: {elapsed_avg_python_time}")
    start_pytorch_time = time.time()
    rgb_to_grayscale_pytorch(input_image)
    end_pytorch_time = time.time()
    elapsed_avg_pytorch_time = (end_pytorch_time - start_pytorch_time)
    print(f"Average PyTorch Time: {elapsed_avg_pytorch_time}")

    # Push the input image to the GPU
    input_image = input_image.to(dtype=torch.uint8,device='cuda')
    # warmup cuda
    for _ in range(cuda_warmup):
        grayscale_module.rgb_to_grayscale_cuda(input_image)
    
    start_cuda_time = torch.cuda.Event(enable_timing=True)
    end_cuda_time = torch.cuda.Event(enable_timing=True)
    
    start_cuda_time.record()
    for _ in range(cuda_runs):
        grayscale_module.rgb_to_grayscale_cuda(input_image)
    end_cuda_time.record()
    torch.cuda.synchronize()
    elapsed_avg_cuda_time = torch.cuda.Event.elapsed_time(start_cuda_time, end_cuda_time)/cuda_runs
    print(f"Average CUDA Time: {elapsed_avg_cuda_time} ms")

    return elapsed_avg_python_time, elapsed_avg_pytorch_time, elapsed_avg_cuda_time
    

In [22]:
python_time, pytorch_time, cuda_time = benchmarking_run(input_image)

Average Python Time: 49.667633056640625
Average PyTorch Time: 0.011578083038330078
Average CUDA Time: 0.1862015962600708 ms


## Observations

Brief observations:

* **Python loop is extremely slow** at ~49.67 s because pixel-by-pixel work is controlled by the Python interpreter.
* **Vectorized PyTorch is dramatically faster** at ~11.58 ms because the computation is pushed into optimized tensor operations rather than Python loops.
* **Custom CUDA is fastest** at ~0.186 ms, roughly **62× faster than the PyTorch version** and about **267,000× faster than the Python loop** for this measurement.
* This shows the main progression clearly: **Python iteration → vectorized tensor operations → explicit GPU parallelism**.
* The CUDA result also fits the lecture’s expectation that grayscale conversion is a simple, highly parallel, mostly memory-oriented workload. 
